Now we want to code an encoder-decoder architecture transformer.

# So how is this different from our decoder-only architecture that we implemented earlier?

In a decoder-only transformer, you have just one sequence going in, and one stack of blocks processing it. An example would be "The cat sat on the" and our target word would be "mat."

Each block ends up doing masked-self attention and feedforward.
Masked-self attention: token at position t can only look at positions before it <= t
x1 attends to x1, x2 attends to x1 and x2, etc.

Decoder architectures are therefore useful for autoregressive generation -> predicting next tokens from previous ones

# Now in an encoder-decoder structure

Encoder will read the entire source sequence all at once. The encoder will use unmasked self-attention so every source token can attend to every other token.
In this case x1 will attend to x1, x2, x3, x4 and same with x2, ...

Decoder will generate the target sequence one token at a time. It has three parts:
1. masked self-attention over previous tokens
2. cross-attention to the encoder output
3. feedforward

Decoder token will look backward at earlier target tokens and also look at the encoded source sentence.

Entire structure goes like this:
source tokens -> encoder -> encoder states
target prefix -> decoder 
decoder output -> LM head

In Decoder-only attention: only self-attention on one sequence. 
In Encoder-decoder attention: two types of attention in the decoder: masked self-attention (same as decoder-only), but also cross-attention. 

What is cross attention?
Decoder can basically ask, "while generating this target token, which source words matter?"
queries comes from decoder, keys/values come from encoder output. 

# Let's look at an example:
Let's suppose we want to do a translation task: "I love cats" -> "Jaime les chats"

In a decoder-only example, we would flatten everything into one stream, model predicts tokens from left to right.

In a encoder-decoder way, the encoder first reads "I love cats" and builds representations for the entire sentence. Then when the decoder generates "Jaime","les","chats", it can directly attend to those encoder representations


In [2]:
# Ok now let's code up this encoder-decoder architecture:

# A lot of things are still the same

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

MASK_VALUE = -1e9
class TokenEmbedder(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, max_len: int):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.positional_embedding = nn.Embedding(max_len, d_model)

    def forward(self, x: Tensor) -> Tensor:
        batch_size, seq_len = x.shape
        pos = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, seq_len)
        return self.token_embedding(x) + self.positional_embedding(pos)


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x: Tensor) -> Tensor:
        x = self.fc1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


class LayerNorm(nn.Module):
    def __init__(self, hidden_size: int, eps: float):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(hidden_size))
        self.beta = nn.Parameter(torch.zeros(hidden_size))
        self.eps = eps

    def forward(self, x: Tensor) -> Tensor:
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_hat = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * x_hat + self.beta


In [3]:
# However, now let's look at our multi-head attention implementation. We will need to modify it to support cross-attention between the encoder and decoder.
# Before we implement the full multi-head attention, let's first implement the scaled dot-product attention function that it will use internally. This function will compute the attention scores and apply the causal mask for the decoder self-attention.
# Again this is the same code that we used in the decoder-only case

class QKVProjection(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_k: int):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_k

        self.q_proj = nn.Linear(d_model, num_heads * d_k)
        self.k_proj = nn.Linear(d_model, num_heads * d_k)
        self.v_proj = nn.Linear(d_model, num_heads * d_k)

    def forward(self, x: Tensor) -> (Tensor, Tensor, Tensor):
        batch_size, seq_len, _ = x.shape
        q = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.num_heads * self.d_k).transpose(1, 2)
        return q, k, v

In [4]:

class MultiHeadAttention(nn.Module):
    '''
    This differs from the decoder-only self-attention in that it can optionally take a separate x_kv input for the keys and values,
    which allows it to perform cross-attention between the decoder queries and encoder keys/values.
    - self-attention: query, key, value all come from x_q
    - cross-attention: query comes from x_q, key/value come from x_kv
    '''
    def __init__(self, d_model: int, num_heads: int, d_k: int, attn_pdrop: float = 0.0):
        super().__init__()
        assert num_heads * d_k == d_model, "d_model must equal num_heads * d_k"

        self.num_heads = num_heads
        self.d_k = d_k
        self.attn_pdrop = attn_pdrop

        self.qkv_projection = QKVProjection(d_model, num_heads, d_k)
        self.out_linear = nn.Linear(d_model, d_model)

    def forward(self, x_q: Tensor, x_kv: Tensor = None, causal_mask: bool = False, query_attention_mask: Tensor = None, key_value_attention_mask: Tensor = None,):

        '''
        x_q: the tensor used to generate queries
        x_kv: optional tensor used to generate keys and values
        causal_mask: whether to prevent looking ahead
        query_attention_mask: mask for padded query positions
        key_value_attention_mask: mask for padded key/value positions
        '''

        # this basically says if x_kv is not provided, then we are doing self-attention and we can just use x_q for keys and values as well.
        # But if x_kv is provided, then we are doing cross-attention and we should use x_kv to generate keys and values.
        if x_kv is None:
            x_kv = x_q

        # only take query output from qkv_projection for x_q, and only take key/value output for x_kv
        Q, _, _ = self.qkv_projection(x_q)
        _, K, V = self.qkv_projection(x_kv)

        # QKVProjection returns V flattened across heads as (B, H * d_k, T),
        # so reshape it back to (B, H, T, d_k) for attention.
        B, _, Tk = V.shape
        V = V.contiguous().view(B, self.num_heads, self.d_k, Tk).transpose(-2, -1)

        # here Tq is sequence length of queries, and Tk is sequence length of keys/values. They can be different in cross-attention.
        B, H, Tq, d_k = Q.shape
        Tk = K.shape[2]

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # causal mask only makes sense when attending left-to-right in decoder self-attention
        # in cross-attention, the queries are from the decoder and the keys/values are from the encoder,
        # so we don't want to apply a causal mask there.
        if causal_mask:
            causal = torch.triu(
                torch.ones(Tq, Tk, device=x_q.device, dtype=torch.bool),
                diagonal=1
            )
            scores = scores.masked_fill(causal, MASK_VALUE)

        # mask out padding positions in keys/values
        if key_value_attention_mask is not None:
            # (B, Tk) -> (B, 1, 1, Tk)
            kv_mask = (key_value_attention_mask == 0).view(B, 1, 1, Tk)
            scores = scores.masked_fill(kv_mask, MASK_VALUE)

        attn_weights = F.softmax(scores, dim=-1)

        if self.attn_pdrop is not None and self.training:
            attn_weights = F.dropout(attn_weights, p=self.attn_pdrop, training=True)

        context = torch.matmul(attn_weights, V)  # (B, H, Tq, d_k)
        context = context.transpose(1, 2).contiguous().view(B, Tq, H * d_k)
        output = self.out_linear(context)

        # output shape is (B, Tq, d_model), which matches the input shape of x_q, so we can add a residual connection if we want.
        return attn_weights, output


In [5]:
# Now let's implement our Encoder Block
# Our encoder block should do two main things
# 1. Self-attention: lets each token look at all other tokens in the input sequence.
# 2. Feedforward network: applies a small MLP independently to each token position.

# structure is x -> LayerNorm -> Self-Attention -> Add & Dropout -> LayerNorm -> FeedForward -> Add & Dropout

class EncoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, attn_pdrop: float, dropout: float, d_ff: int, eps: float):
        super().__init__()
        d_k = d_model // num_heads

        self.self_attn = MultiHeadAttention(d_model, num_heads, d_k, attn_pdrop)
        self.ff = FeedForward(d_model, d_ff, dropout)

        self.ln1 = LayerNorm(d_model, eps)
        self.ln2 = LayerNorm(d_model, eps)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: Tensor, attention_mask: Tensor = None) -> Tensor:
        x_norm = self.ln1(x)
        _, attn_out = self.self_attn(x_q=x_norm, x_kv=x_norm, causal_mask=False, query_attention_mask=attention_mask, key_value_attention_mask=attention_mask)
        x = x + self.dropout(attn_out)
        x_norm = self.ln2(x)
        ff_out = self.ff(x_norm)
        x = x + self.dropout(ff_out)

        return x


In [6]:
# Now we can implement the decoder block

# This decoder block has three main components:
# 1. Masked self-attention: lets each position in the decoder look at previous positions in the decoder output so far (causal mask prevents looking ahead).
# 2. Cross-attention: lets each position in the decoder attend to all positions in the encoder output, which allows the decoder to gather information from the input sequence.
# 3. Feedforward network: applies a small MLP independently to each token position in the decoder output.

class DecoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, attn_pdrop: float, dropout: float, d_ff: int, eps: float):
        super().__init__()
        d_k = d_model // num_heads

        # decoder self attention
        self.self_attn = MultiHeadAttention(d_model, num_heads, d_k, attn_pdrop)
        # cross attention
        self.cross_attn = MultiHeadAttention(d_model, num_heads, d_k, attn_pdrop)

        self.ff = FeedForward(d_model, d_ff, dropout)

        self.ln1 = LayerNorm(d_model, eps)
        self.ln2 = LayerNorm(d_model, eps)
        self.ln3 = LayerNorm(d_model, eps)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: Tensor, encoder_output: Tensor, decoder_attention_mask: Tensor = None, encoder_attention_mask: Tensor = None) -> Tensor:
        # masked self-attention
        # x is the decoder input, which is used to generate queries, keys, and values for the masked self-attention.
        # encoder_output is the output from the encoder, which will be used as keys and values for the cross-attention.
        # decoder_attention_mask is used to mask out padding positions in the decoder input for both the masked self-attention and the cross-attention.
        # encoder_attention_mask is used to mask out padding positions in the encoder output for the cross-attention.
        x_norm = self.ln1(x)
        _, self_attn_out = self.self_attn(x_q=x_norm, x_kv=x_norm, causal_mask=True, query_attention_mask=decoder_attention_mask, key_value_attention_mask=decoder_attention_mask)
        x = x + self.dropout(self_attn_out)

        # cross-attention: decoder queries attend to encoder keys/values
        # here the main difference is that we pass encoder_output as x_kv to the cross-attention, which allows the decoder queries to attend to the encoder keys/values.
        x_norm = self.ln2(x)
        _, cross_attn_out = self.cross_attn(x_q=x_norm, x_kv=encoder_output, causal_mask=False, query_attention_mask=decoder_attention_mask, key_value_attention_mask=encoder_attention_mask)
        x = x + self.dropout(cross_attn_out)

        # feedforward
        x_norm = self.ln3(x)
        ff_out = self.ff(x_norm)
        x = x + self.dropout(ff_out)

        return x


In [7]:
# Now we can stack multiple encoder blocks to create the full encoder, and stack multiple decoder blocks to create the full decoder.

class Encoder(nn.Module):
    def __init__(self, num_layers: int, d_model: int, num_heads: int, attn_pdrop: float, dropout: float, d_ff: int, eps: float):
        super().__init__()
        self.layers = nn.ModuleList()
        for _ in range(num_layers):
            self.layers.append(EncoderBlock(d_model, num_heads, attn_pdrop, dropout, d_ff, eps))

    def forward(self, x: Tensor, attention_mask: Tensor = None) -> Tensor:
        for layer in self.layers:
            x = layer(x, attention_mask=attention_mask)
        return x


class Decoder(nn.Module):
    def __init__(self, num_layers: int, d_model: int, num_heads: int, attn_pdrop: float, dropout: float, d_ff: int, eps: float):
        super().__init__()
        self.layers = nn.ModuleList()
        for _ in range(num_layers):
            self.layers.append(DecoderBlock(d_model, num_heads, attn_pdrop, dropout, d_ff, eps))

    def forward(self, x: Tensor, encoder_output: Tensor, decoder_attention_mask: Tensor = None, encoder_attention_mask: Tensor = None) -> Tensor:
        for layer in self.layers:
            x = layer(x, encoder_output, decoder_attention_mask=decoder_attention_mask, encoder_attention_mask=encoder_attention_mask)
        return x

In [8]:
# Finally, we can put everything together into an Encoder-Decoder Transformer model.
# This is the full encoder-decoder structure

# a source embedding layer
# an encoder
# a target embedding layer
# a decoder
# a final linear layer that turns decoder outputs into vocabulary scores


class EncoderDecoderTransformer(nn.Module):
    def __init__(self, src_vocab_size: int, tgt_vocab_size: int, d_model: int, num_heads: int, attn_pdrop: float, dropout: float, d_ff: int, max_len: int, num_layers: int, eps: float):
        super().__init__()

        self.src_embedder = TokenEmbedder(src_vocab_size, d_model, max_len)
        self.tgt_embedder = TokenEmbedder(tgt_vocab_size, d_model, max_len)

        self.encoder = Encoder(num_layers, d_model, num_heads, attn_pdrop, dropout, d_ff, eps)
        self.decoder = Decoder(num_layers, d_model, num_heads, attn_pdrop, dropout, d_ff, eps)
        self.lm_head = nn.Linear(d_model, tgt_vocab_size, bias=False)

    def forward(self, src_tokens: Tensor, tgt_tokens: Tensor, encoder_attention_mask: Tensor = None,decoder_attention_mask: Tensor = None) -> Tensor:
        src_hidden = self.src_embedder(src_tokens)
        encoder_output = self.encoder(src_hidden, attention_mask=encoder_attention_mask)

        tgt_hidden = self.tgt_embedder(tgt_tokens)
        decoder_output = self.decoder(tgt_hidden, encoder_output, decoder_attention_mask=decoder_attention_mask, encoder_attention_mask=encoder_attention_mask)

        # at the very end, we can just take a linear layer on top of the decoder output to get logits for each token in the target vocabulary.
        logits = self.lm_head(decoder_output)
        return logits


In [9]:
# Example training loop on a tiny toy translation dataset.
# This is intentionally small so the model can overfit quickly and we can inspect the decoded output.

from torch.utils.data import DataLoader, Dataset
# training data, each item is a source sentence in ENGLISH and a target sentence in FRENCH

pairs = [
    ("i love cats", "j aime les chats"),
    ("i love dogs", "j aime les chiens"),
    ("you love cats", "tu aimes les chats"),
    ("you love dogs", "tu aimes les chiens"),
    ("we eat apples", "nous mangeons des pommes"),
    ("we eat bread", "nous mangeons du pain"),
    ("they eat apples", "ils mangent des pommes"),
    ("they eat bread", "ils mangent du pain"),
    ("the cat sleeps", "le chat dort"),
    ("the dog sleeps", "le chien dort"),
    ("the cat runs", "le chat court"),
    ("the dog runs", "le chien court"),
]
# we also have special tokens for padding, beginning of sentence, end of sentence, and unknown tokens that are not in the vocabulary.
special_tokens = ["<pad>", "<bos>", "<eos>", "<unk>"]

def build_vocab(sentences):
    vocab = {tok: idx for idx, tok in enumerate(special_tokens)}
    for sentence in sentences:
        for token in sentence.split():
            if token not in vocab:
                vocab[token] = len(vocab)
    return vocab

# build separate vocabularies for source and target languages, since they have different tokens.
# We also create reverse vocabularies (src_itos and tgt_itos) to decode output ids back to tokens later when we want to inspect the generated translations.
src_vocab = build_vocab([src for src, _ in pairs])
tgt_vocab = build_vocab([tgt for _, tgt in pairs])
# reverse vocab for decoding output ids back to tokens
src_itos = {idx: tok for tok, idx in src_vocab.items()}
tgt_itos = {idx: tok for tok, idx in tgt_vocab.items()}

print("Source Vocabulary:", src_vocab)
print("Target Vocabulary:", tgt_vocab)
print("Source Itos:", src_itos)
print("Target Itos:", tgt_itos)


Source Vocabulary: {'<pad>': 0, '<bos>': 1, '<eos>': 2, '<unk>': 3, 'i': 4, 'love': 5, 'cats': 6, 'dogs': 7, 'you': 8, 'we': 9, 'eat': 10, 'apples': 11, 'bread': 12, 'they': 13, 'the': 14, 'cat': 15, 'sleeps': 16, 'dog': 17, 'runs': 18}
Target Vocabulary: {'<pad>': 0, '<bos>': 1, '<eos>': 2, '<unk>': 3, 'j': 4, 'aime': 5, 'les': 6, 'chats': 7, 'chiens': 8, 'tu': 9, 'aimes': 10, 'nous': 11, 'mangeons': 12, 'des': 13, 'pommes': 14, 'du': 15, 'pain': 16, 'ils': 17, 'mangent': 18, 'le': 19, 'chat': 20, 'dort': 21, 'chien': 22, 'court': 23}
Source Itos: {0: '<pad>', 1: '<bos>', 2: '<eos>', 3: '<unk>', 4: 'i', 5: 'love', 6: 'cats', 7: 'dogs', 8: 'you', 9: 'we', 10: 'eat', 11: 'apples', 12: 'bread', 13: 'they', 14: 'the', 15: 'cat', 16: 'sleeps', 17: 'dog', 18: 'runs'}
Target Itos: {0: '<pad>', 1: '<bos>', 2: '<eos>', 3: '<unk>', 4: 'j', 5: 'aime', 6: 'les', 7: 'chats', 8: 'chiens', 9: 'tu', 10: 'aimes', 11: 'nous', 12: 'mangeons', 13: 'des', 14: 'pommes', 15: 'du', 16: 'pain', 17: 'ils', 18:

In [10]:
PAD_ID = tgt_vocab["<pad>"]
BOS_ID = tgt_vocab["<bos>"]
EOS_ID = tgt_vocab["<eos>"]
SRC_PAD_ID = src_vocab["<pad>"]

# here we write two functions, one to encode source sentences into lists of token ids using the source vocabulary,
# and another to encode target sentences into lists of token ids using the target vocabulary.
# We also add special tokens like <eos> at the end of source sentences and <bos> at the beginning and <eos> at the end of target sentences to indicate the start and end of the sequences.

def encode_src(text):
    tokens = []
    for tok in text.split():
        if tok not in src_vocab:
            # print(f"Warning: token '{tok}' not in source vocabulary, using <unk> token id")
            tokens.append(src_vocab["<unk>"])
        else:
            tokens.append(src_vocab[tok])

    return tokens + [src_vocab["<eos>"]]

def encode_tgt(text):
    tokens = []
    for tok in text.split():
        if tok not in tgt_vocab:
            # print(f"Warning: token '{tok}' not in target vocabulary, using <unk> token id")
            tokens.append(tgt_vocab["<unk>"])
        else:
            tokens.append(tgt_vocab[tok])
    return [BOS_ID] + tokens + [EOS_ID]


In [16]:
# Now we can create a PyTorch Dataset to hold our translation pairs, and a DataLoader to batch them together during training.
# The collate function will pad the source and target sequences in each batch to the same length and create the appropriate attention masks for the encoder and decoder.

class TranslationDataset(Dataset):
    def __init__(self, pairs):
        self.examples = []
        for src, tgt in pairs:
            self.examples.append((encode_src(src), encode_tgt(tgt)))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

# okay so what exactly is the collate function doing here? Let's break it down step by step.
def collate_batch(batch):

    # we first unzip the batch of examples into separate lists of source and target token id sequences.
    # Each item in src_batch is a list of token ids for a source sentence,
    # and each item in tgt_batch is a list of token ids for a target sentence.
    src_batch, tgt_batch = zip(*batch)

    # print("src_batch:", src_batch)
    # print("tgt_batch:", tgt_batch)

    # we want to find the max length of the source and target sequences in this batch so we can pad them to the same length.
    src_max_len = max(len(x) for x in src_batch)
    tgt_max_len = max(len(x) for x in tgt_batch)

    # print("src_max_len:", src_max_len)
    # print("tgt_max_len:", tgt_max_len)

    # here we use torch.full to create tensors filled with the padding token id,
    # with shape (batch_size, max_seq_len) for both source and target.
    # this will basically create a 3D matrix
    src_tokens = torch.full((len(batch), src_max_len), SRC_PAD_ID, dtype=torch.long)
    tgt_tokens = torch.full((len(batch), tgt_max_len), PAD_ID, dtype=torch.long)

    # now we iterate over each example in the batch and copy the actual token ids
    # into the appropriate positions in the src_tokens and tgt_tokens tensors, leaving the rest as padding.
    for i, (src_ids, tgt_ids) in enumerate(zip(src_batch, tgt_batch)):
        src_tokens[i, :len(src_ids)] = torch.tensor(src_ids)
        tgt_tokens[i, :len(tgt_ids)] = torch.tensor(tgt_ids)
    # this means that shorter sequences will be padded with the PAD_ID token at the end

    # this encoder_attention mask will just check where the src_tokens are not equal to the SRC_PAD_ID,
    # which means those positions are actual tokens and not padding.
    # It will be a binary mask of shape (batch_size, src_max_len) where 1 indicates a real token and 0 indicates padding.
    encoder_attention_mask = []

    for sequence in src_tokens:
        row = []
        for tok in sequence:
            if tok.item() != SRC_PAD_ID:
                row.append(1)
            else:
                row.append(0)
        encoder_attention_mask.append(row)

    encoder_attention_mask = torch.tensor(encoder_attention_mask).long()
    # print("encoder_attention_mask:", encoder_attention_mask)

    # now we prepare the decoder input and target sequences for teacher forcing during training.
    # the decoder input is the target sequence shifted to the right by one position, with a BOS token at the beginning,
    # and the decoder target is the target sequence shifted to the left by one
    # print("tgt_tokens:", tgt_tokens)
    decoder_input = tgt_tokens[:, :-1]
    decoder_target = tgt_tokens[:, 1:]
    # print("decoder_input:", decoder_input)
    # print("decoder_target:", decoder_target)

    # we prepare our decoder_attention_mask in a similar way to the encoder_attention_mask,
    # but this time we check where the decoder_input is not equal to the PAD_ID,
    # since the decoder input is what we will feed into the model during training.
    decoder_attention_mask = []
    for sequence in decoder_input:
        row = []
        for tok in sequence:
            if tok.item() != PAD_ID:
                row.append(1)
            else:
                row.append(0)
        decoder_attention_mask.append(row)

    decoder_attention_mask = torch.tensor(decoder_attention_mask).long()

    return src_tokens, decoder_input, decoder_target, encoder_attention_mask, decoder_attention_mask

dataset = TranslationDataset(pairs)
loader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_batch)


In [17]:
# we can now create an instance of our EncoderDecoderTransformer model,
# define a loss function and an optimizer, and run a training loop over our tiny translation dataset.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = EncoderDecoderTransformer(
    src_vocab_size=len(src_vocab),
    tgt_vocab_size=len(tgt_vocab),
    d_model=64,
    num_heads=4,
    attn_pdrop=0.1,
    dropout=0.1,
    d_ff=128,
    max_len=16,
    num_layers=2,
    eps=1e-6,
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)

num_epochs = 200
model.train()
for epoch in range(num_epochs):
    total_loss = 0.0
    for src_tokens, decoder_input, decoder_target, encoder_attention_mask, decoder_attention_mask in loader:

        src_tokens = src_tokens.to(device)
        decoder_input = decoder_input.to(device)
        decoder_target = decoder_target.to(device)
        encoder_attention_mask = encoder_attention_mask.to(device)
        decoder_attention_mask = decoder_attention_mask.to(device)

        optimizer.zero_grad()
        # here we pass the source tokens, decoder input, and attention masks into the model to get the output logits.
        # the output logits should be in the shape, (batch_size, tgt_seq_len, tgt_vocab_size),
        # which gives us the predicted scores for each token in the target vocabulary at each position in the target sequence.
        logits = model(src_tokens, decoder_input, encoder_attention_mask=encoder_attention_mask, decoder_attention_mask=decoder_attention_mask)
        # now we want to flatten our logits from (B, T, V) to (B * T, V)
        # and our decoder_target from (B, T) to (B * T) so we can compute the cross-entropy loss across all tokens in the batch at once.
        loss = criterion(logits.reshape(-1, logits.size(-1)), decoder_target.reshape(-1))
        # backpropagate the loss and update the model parameters
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 25 == 0:
        print(f"epoch {epoch + 1:3d} | loss {total_loss / len(loader):.4f}")


epoch  25 | loss 0.0013
epoch  50 | loss 0.0002
epoch  75 | loss 0.7681
epoch 100 | loss 0.0739
epoch 125 | loss 0.0041
epoch 150 | loss 0.0081
epoch 175 | loss 0.0002
epoch 200 | loss 0.0001


In [18]:
# now let's see if our model can generate translations for some of the input sentences after training.
# we define a function called greedy_decode that takes a source sentence as input and generates a translation by
# greedily selecting the most likely token at each step until it generates an EOS token or reaches a maximum length.
def greedy_decode(model, src_sentence, max_new_tokens=10):
    model.eval()
    # we first want to encode the source sentence into a tensor of token ids using our source vocabulary,
    # and create the appropriate attention mask for the encoder.
    src_ids = encode_src(src_sentence)
    src_tokens = torch.tensor(src_ids, dtype=torch.long, device=device).unsqueeze(0)
    encoder_attention_mask = (src_tokens != SRC_PAD_ID).long()

    generated = [BOS_ID]
    for _ in range(max_new_tokens):
        decoder_input = torch.tensor(generated, dtype=torch.long, device=device).unsqueeze(0)
        decoder_attention_mask = (decoder_input != PAD_ID).long()

        # here we pass the source tokens, decoder input, and attention masks into the model to get the output logits
        # for the next token prediction. This allows the next token to be predicted from the source sentence and all previously generated tokens in the decoder input.
        with torch.no_grad():
            logits = model(src_tokens, decoder_input, encoder_attention_mask=encoder_attention_mask, decoder_attention_mask=decoder_attention_mask)

        # then we take the logits for the last position in the decoder output (which corresponds to the next token prediction) and
        # select the token with the highest score as our next token.
        # here logits[0, -1] stands for batch element 0 and last generated position
        # this is also why this is called greedy decode because we don't do beam search or sampling, we just take the single most likely token at each step.
        next_token = logits[0, -1].argmax(dim=-1).item()
        generated.append(next_token)
        if next_token == EOS_ID:
            break
    # now we need to convert the generated token ids back into tokens using our target vocabulary,
    # and join them together into a string to get the final generated translation.
    decoded = []
    # we want to skip BOS token and EOS token
    for token_id in generated[1:]:
        if token_id == EOS_ID:
            break
        decoded.append(tgt_itos[token_id])
    return " ".join(decoded)


In [ ]:
# Finally, we can test with some test sentences
test_sentences = [
    "i love cats",
    "you love dogs",
    "the cat sleeps",
    "they eat bread",
]

for sentence in test_sentences:
    prediction = greedy_decode(model, sentence)
    print(f"{sentence} -> {prediction}")


i love cats -> j aime les chats
you love dogs -> tu aimes les chiens
the cat sleeps -> le chat dort
they eat bread -> ils mangent du pain
